TPU and files location

In [1]:
# # detect and init the TPU
# tpu = tf.distribute.cluster_resolver.TPUClusterResolver()
# tf.config.experimental_connect_to_cluster(tpu)
# tf.tpu.experimental.initialize_tpu_system(tpu)

# # instantiate a distribution strategy
# tpu_strategy = tf.distribute.experimental.TPUStrategy(tpu)

In [2]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# # You can write up to 5GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

Librairies

In [3]:
import tensorflow as tf
import pandas as pd
import numpy as np
import os
import random
import pickle

from tensorflow import keras as K
from tensorflow.keras import layers as L


2026-04-28 03:16:09.226541: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-28 03:16:09.236783: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777346169.247720      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777346169.251087      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-28 03:16:09.263550: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

Set seed

In [4]:
def seed_all(seed = 20):
    # 1. Set `PYTHONHASHSEED` environment variable at a fixed value: import os os.environ['PYTHONHASHSEED']=str(seed_value) 
    os.environ['PYTHONHASHSEED']=str(seed)
    # 2. Set `python` built-in pseudo-random generator at a fixed value: import random random.seed(seed_value)
    random.seed(seed) 
    # 3. Set `numpy` pseudo-random generator at a fixed value:
    np.random.seed(seed) 
    # 4. Set `tensorflow` pseudo-random generator at a fixed value: import tensorflow as tf tf.set_random_seed(seed_value)
    tf.random.set_seed(seed)
    # # 5. For layers that introduce randomness like dropout, make sure to set seed values:
    # model.add(Dropout(0.25, seed=seed_value))
    # #6 Configure a new global `tensorflow` session: 
    # from keras import backend as K 
    # session_conf = tf.ConfigProto(intra_op_parallelism_threads=1, inter_op_parallelism_threads=1) 
    # sess = tf.Session(graph=tf.get_default_graph(), config=session_conf)
    # K.set_session(sess)
    
seed_all(20)

In [5]:
#Import raw data and copy
# raw_train = pd.read_csv('/kaggle/input/osic-pulmonary-fibrosis-progression/train.csv')
raw_test = pd.read_csv('/kaggle/input/osic-pulmonary-fibrosis-progression/test.csv')
X_prediction = pd.read_csv('/kaggle/input/osic-pulmonary-fibrosis-progression/sample_submission.csv')

# size_image = pd.read_csv('/kaggle/input/prep-data/size_image.csv')
# list_files = pd.read_csv('/kaggle/input/prep-data/list_files.csv')

train = pd.read_csv('/kaggle/input/prep-data/train.csv')

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/prep-data/train.csv'

In [6]:
ID='Patient_Week'
PINBALL_QUANTILE = [0.2, 0.50, 0.8]
LAMBDA_LOSS = 0.8
selected_columns = ['Base_week', 'Base_FVC', 'Base_percent', 'Age', 'Sex', 'Weeks', '_Currently smokes', '_Ex-smoker', '_Never smoked']

Create base variable and expand the dataset

In [7]:
# rename_cols = {'Weeks_x':'Base_week', 'FVC_x': 'Base_FVC', 'Percent_x': 'Base_percent', 'Age_x': 'Age', 'SmokingStatus_x': 'SmokingStatus', 'Sex_x':'Sex', 'Weeks_y':'Weeks', 'FVC_y': 'FVC'}
# drop_cols = ['Age_y', 'Sex_y', 'SmokingStatus_y', 'Percent_y']
# train = train.merge(train, how='left', left_on='Patient', right_on='Patient').rename(columns=rename_cols).drop(columns=drop_cols)
# train[ID] = train['Patient'].astype(str) + '_' + train['Weeks'].astype(str)
# train = train[['Patient', 'Base_week', 'Base_FVC', 'Base_percent', 'Age', 'Sex', 'SmokingStatus', 'Weeks', 'Patient_Week', 'FVC']].reset_index(drop=True)
# #Keep base week atm
# # train = train[train['Weeks']!=train['Base_week']]

In [8]:
from sklearn.preprocessing import OneHotEncoder as SklearnOneHotEncoder

#Override OneHotEncoder to have the column names created automatically (Ex-smoker, Never Smoked...) 
class OneHotEncoder(SklearnOneHotEncoder):
    def __init__(self, **kwargs):
        super(OneHotEncoder, self).__init__(**kwargs)
        self.fit_flag = False

    def fit(self, X, **kwargs):
        out = super().fit(X)
        self.fit_flag = True
        return out

    def transform(self, X, categories, index='', name='', **kwargs):
        sparse_matrix = super(OneHotEncoder, self).transform(X)
        new_columns = self.get_new_columns(X=X, name=name, categories=categories)
        d_out = pd.DataFrame(sparse_matrix.toarray(), columns=new_columns, index=index)
        return d_out

    def fit_transform(self, X, categories, index, name, **kwargs):
        self.fit(X)
        return self.transform(X, categories=categories, index=index, name=name)

    def get_new_columns(self, X, name, categories):
        new_columns = []
        for j in range(len(categories)):
            new_columns.append('{}_{}'.format(name, categories[j]))
        return new_columns
    
    
from sklearn.preprocessing import LabelEncoder
from sklearn.exceptions import NotFittedError

class data_preparation():
    def __init__(self):
        self.enc_sex = LabelEncoder()
        self.enc_smok = LabelEncoder()
        self.onehotenc_smok = OneHotEncoder()
        
        
    def __call__(self, data_untransformed):
        data = data_untransformed.copy(deep=True)
        
        #For the test set/Already fitted
        try:
            data['Sex'] = self.enc_sex.transform(data['Sex'].values)
            data['SmokingStatus'] = self.enc_smok.transform(data['SmokingStatus'].values)
            data = pd.concat([data.drop(columns=['SmokingStatus']), self.onehotenc_smok.transform(data['SmokingStatus'].values.reshape(-1,1), categories=self.enc_smok.classes_, name='', index=data.index).astype(int)], axis=1)
            
        #For the train set/Not yet fitted    
        except NotFittedError:
            data['Sex'] = self.enc_sex.fit_transform(data['Sex'].values)
            data['SmokingStatus'] = self.enc_smok.fit_transform(data['SmokingStatus'].values)
            data = pd.concat([data.drop(columns=['SmokingStatus']), self.onehotenc_smok.fit_transform(data['SmokingStatus'].values.reshape(-1,1), categories=self.enc_smok.classes_, name='', index=data.index).astype(int)], axis=1)
            
        return data

Create base variable for prediction

In [9]:
X_prediction['Patient'] = X_prediction['Patient_Week'].str.extract(r'(.*)_.*')
X_prediction['Weeks'] = X_prediction['Patient_Week'].str.extract(r'.*_(.*)').astype(int)
X_prediction = X_prediction[['Patient', 'Weeks', 'Patient_Week']]
rename_cols = {'Weeks_y':'Base_week', 'Weeks_x': 'Weeks', 'Percent':'Base_percent', 'FVC':'Base_FVC'}
X_prediction = X_prediction.merge(raw_test, how='left', left_on='Patient', right_on='Patient').rename(columns=rename_cols)[['Patient', 'Base_week', 'Base_FVC', 'Base_percent', 'Age', 'Sex', 'SmokingStatus', 'Weeks', 'Patient_Week']].reset_index(drop=True)

In [10]:
pickefile = open('/kaggle/input/prep-data/data_prep', 'rb')
data_prep = pickle.load(pickefile)
pickefile.close()
X_prediction = data_prep(X_prediction).sort_values('Patient')

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/prep-data/data_prep'

In [11]:
C1, C2 = tf.constant(70, dtype='float32'), tf.constant(1000, dtype="float32")
#=============================#
def score(y_true, y_pred):
    y_true=tf.dtypes.cast(y_true, tf.float32)
    y_pred=tf.dtypes.cast(y_pred, tf.float32)
    sigma = y_pred[:, 2] - y_pred[:, 0]
    fvc_pred = y_pred[:, 1]
    
    #sigma_clip = sigma + C1
    sigma_clip = tf.maximum(sigma, C1)
    delta = tf.abs(y_true[:, 0] - fvc_pred)
    delta = tf.minimum(delta, C2)
    sq2 = tf.sqrt( tf.dtypes.cast(2, dtype=tf.float32) )
    metric = (delta / sigma_clip)*sq2 + tf.math.log(sigma_clip* sq2)
    return K.backend.mean(metric)
#============================#
def qloss(y_true, y_pred):
    # Pinball loss for multiple quantiles
    qs = PINBALL_QUANTILE
    q = tf.constant(np.array([qs]), dtype=tf.float32)
    e = y_true - y_pred
    v = tf.maximum(q*e, (q-1)*e)
    return K.backend.mean(v)
#=============================#
def mloss(_lambda):
    def loss(y_true, y_pred):
        return _lambda * qloss(y_true, y_pred) + (1 - _lambda)*score(y_true, y_pred)
    return loss

2026-04-28 03:16:16.516552: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [12]:
def create_model():
    model_input = K.Input(shape=(9,))
    x = L.Dense(500, activation="relu")(model_input)
    x = L.Dense(200, activation="relu")(x)
    x = L.Dense(100, activation="relu")(x)
    FVC = L.Dense(3, activation="relu")(x)
#     p1 = L.Dense(3, activation="linear", name="p1")(x)
#     p2 = L.Dense(3, activation="relu", name="p2")(x)
#     FVC = L.Lambda(lambda x: x[0] + tf.cumsum(x[1], axis=1), 
#                          name="FVC")([p1, p2])

    model = K.Model(
        inputs=model_input,
        outputs=[FVC],
    )
    model.compile(
        optimizer=K.optimizers.RMSprop(),
        loss=mloss(LAMBDA_LOSS),
    )
    return model

model = create_model()
# tf.keras.utils.plot_model(model)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 9)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 500)            │         5,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 200)            │       100,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 100)            │        20,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 3)              │           303 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 125,603 (490.64 KB)

 Trainable params: 125,603 (490.64 KB)

 Non-trainable params: 0 (0.00 B)

In [13]:
SELECTED_COLUMNS = ['Weeks', 'Base_week', 'Base_FVC', 'Base_percent', 'Age', 'Sex', '_Currently smokes', '_Ex-smoker', '_Never smoked']
test = train[train['Patient'].isin(['ID00007637202177411956430', 'ID00009637202177434476278'])].copy()
train = train[~train['Patient'].isin(['ID00007637202177411956430', 'ID00009637202177434476278'])]

NameError: name 'train' is not defined

In [14]:
history = model.fit(x=train[SELECTED_COLUMNS], y=train[['FVC']], validation_data=(test[SELECTED_COLUMNS], test[['FVC']]), epochs=100)

NameError: name 'train' is not defined

In [15]:
y_prediction = model.predict(X_prediction[SELECTED_COLUMNS])
sub = pd.DataFrame(data={'Patient_Week':X_prediction['Patient_Week'], 'FVC':y_prediction[:,1], 'Confidence': (y_prediction[:,2]-y_prediction[:,0])})
sub

KeyError: "['_Currently smokes', '_Ex-smoker', '_Never smoked'] not in index"

In [16]:
sub.to_csv('submission.csv', index=False)

NameError: name 'sub' is not defined

In [17]:
# import pydicom
# import matplotlib.pyplot as plt
# img = pydicom.dcmread('/kaggle/input/osic-pulmonary-fibrosis-progression/train/ID00123637202217151272140/137.dcm')
# plt.style.use('grayscale')
# plt.imshow(img.pixel_array)
# img

In [18]:
# VGG = K.applications.VGG16(include_top=False, weights="imagenet", input_tensor=None, input_shape=None, pooling=None)

In [19]:
# VGG.predict(img.pixel_array.reshape(1, 1, img.pixel_array.shape[0], img.pixel_array.shape[1]))

In [20]:
# (img.pixel_array.reshape(1, 1, img.pixel_array.shape[0], img.pixel_array.shape[1])*3).shape